# Encrypted RAG with LangChain-enVector

This example demonstrates the complete workflow of the enVector Python SDK, showcasing its capabilities for Encrypted Retrieval-Augmented Generation (Encrypted RAG) using fully homomorphic encryption (FHE). 
In this example, we'll see:

- How text data is stored and encrypted in the index for RAG
- How the encrypted similarity search is performed with FHE
- How the LLM (Ollama using gemma) leverages RAG while keeping results encrypted until decryption 


## Prerequisites
- enVector server reachable from this notebook environment
- Registered key path and key ID for the target index
- `pyenvector`, `langchain`, `langchain-community`, and `langchain-text-splitters` packages installed
- A PDF document accessible from the working directory

### Import langchain-envector

Import `langchain_envector` to use enVector with the LangChain framework.

In [ ]:
# !pip install langchain-envector langchain_community pypdf langchain-text-splitters

In [ ]:
import langchain_envector

First, load a sample document to search.

In this example, we use a NIST report. This report evaluates how accurate and reliable common empirical formulas are when used to predict fire behavior in various scenarios. For more details about the report, see [NIST Report](https://www.nist.gov/publications/verification-and-validation-commonly-used-empirical-correlations-fire-scenarios) and download the PDF from [Link](https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.1169.pdf).

In [ ]:
import os
from pathlib import Path

!wget -O NIST_SP_1169.pdf "https://nvlpubs.nist.gov/nistpubs/SpecialPublications/NIST.SP.1169.pdf"

PDF_PATH = Path("./NIST_SP_1169.pdf")  # Update with your PDF path
assert PDF_PATH.exists(), f"PDF file not found: {PDF_PATH}"

## Configure connection and embeddings
Fill in your enVector connection details and choose an embedding model. The model dimension must match the index dimension.

In [ ]:
address = os.getenv("ENVECTOR_ADDRESS", "0.0.0.0:50050")
token = os.getenv("ENVECTOR_ACCESS_TOKEN", "")

key_path = "./keys"
key_id = "langchain_key"
index_name = "pdf_demo"
embedding_model = "all-minilm:l6-v2"

print(f"Using key '{key_id}'")
print(f"Using index '{index_name}'")
print(f"Embedding model: {embedding_model}")


## Load the PDF and split into chunks
We rely on LangChain community loaders and text splitters to turn the PDF pages into retrieval-friendly passages.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(str(PDF_PATH))
raw_docs = loader.load()
print(f"Loaded {len(raw_docs)} pages from {PDF_PATH.name}")

# Some embedding models served by Ollama have relatively small context windows.
# Keep chunks comfortably below typical limits to avoid 400s from /api/embed.
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunked_docs = splitter.split_documents(raw_docs)
print(f"Created {len(chunked_docs)} chunks")

## Prepare text and metadata payloads
enVector expects parallel lists of texts and metadata dictionaries. Here we keep track of the original page number for traceability.

In [ ]:
texts = []
metadatas = []
for doc in chunked_docs:
    texts.append(doc.page_content)
    meta = dict(doc.metadata)
    meta.setdefault("source", PDF_PATH.name)
    metadatas.append(meta)

print(texts[0][:200])
print(metadatas[0])
print(f"Prepared {len(texts)} text chunks")

## Set embedding model

In this example, we use Ollama embeddings to convert our text chunks into numerical vectors that can be encrypted and searched.
The embeddings model will transform each text chunk into a high-dimensional vector that captures semantic meaning.
These vectors will be encrypted before being stored in the enVector index.

In [ ]:

import json
import os
import requests
from types import SimpleNamespace
from typing import List

OLLAMA_API_URL = os.getenv("OLLAMA_API_URL", "http://localhost:11434/api")

def _ensure_model_available(model_name: str) -> None:
    try:
        tags = requests.get(f"{OLLAMA_API_URL}/tags").json().get("models", [])
    except requests.RequestException as exc:
        raise RuntimeError("Failed to query Ollama for available models") from exc
    if any(m.get("name") == model_name for m in tags):
        return

    pull_resp = requests.post(
        f"{OLLAMA_API_URL}/pull",
        json={"model": model_name},
        stream=True,
    )
    try:
        for line in pull_resp.iter_lines():
            if not line:
                continue
            payload = json.loads(line.decode("utf-8"))
            if payload.get("status") == "success":
                break
        pull_resp.raise_for_status()
    finally:
        pull_resp.close()

def _ollama_embed(texts: List[str], model_name: str, batch_size: int = 32) -> List[List[float]]:
    _ensure_model_available(model_name)
    embeddings: List[List[float]] = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = requests.post(
            f"{OLLAMA_API_URL}/embed",
            json={
                "model": model_name,
                "input": batch,
            },
        )
        response.raise_for_status()
        embeddings.extend(response.json()["embeddings"])
    return embeddings

def embed_query(text: str, model_name: str) -> List[float]:
    return _ollama_embed([text], model_name)[0]

def embed_documents(texts: List[str], model_name: str) -> List[List[float]]:
    if not texts:
        return []
    return _ollama_embed(texts, model_name)

embeddings = SimpleNamespace(
    embed_query=lambda text: embed_query(text, embedding_model),
    embed_documents=lambda texts: embed_documents(texts, embedding_model),
)
vector_dim = len(embeddings.embed_query("envector probe"))
print(f"Embedding dimension: {vector_dim}")


## Initialize the enVector store
Configure the encrypted vector index and instantiate the LangChain-compatible store. The embedding model derives the vector dimension automatically.

Initialization step includes:
1. `ConnectionConfig`: establishing a connection to the enVector server, 
2. `IndexSettings`: configuring index settings necessary for vector search, including query and metadata encryption, and
3. `KeyConfig`: registering evaluation keys to enable the enVector server to perform secure operations.

In [ ]:
from langchain_envector.config import ConnectionConfig, EnvectorConfig, IndexSettings, KeyConfig
from langchain_envector.vectorstore import Envector

config = EnvectorConfig(
    connection=ConnectionConfig(address=address, access_token=token) if token else ConnectionConfig(address=address),
    key=KeyConfig(key_path=key_path, key_id=key_id, preset="ip1", eval_mode="mm"),
    index=IndexSettings(index_name=index_name, dim=vector_dim),
    create_if_missing=True,
)
store = Envector(config=config, embeddings=embeddings)

## Insert chunks (batched)


In [ ]:
_ = store.add_texts(texts, metadatas)

### Encrypted search on the index

Let's perform an encrypted similarity search using LangChain-enVector.

The enVector vectorstore provides a simple interface through LangChain to perform similarity search on encrypted data.
Under the hood, enVector handles all the encryption, decryption, and secure search operations automatically.
When we call `similarity_search()`, the query is encrypted, the secure similarity search is performed on the encrypted vectors,
and the results are decrypted before being returned.

The `store.similarity_search()` method returns the top-k most relevant documents along with their similarity scores,
making it easy to build secure RAG applications without having to manage encryption directly.

In [ ]:
query = "Which organizations collaborated on NIST SP 1169’s fire model verification and validation study, and what larger NRC report summarizes the results?"

# Query in plaintext
results = store.similarity_search(query, k=3)
for idx, doc in enumerate(results, start=1):
    print(f"--- Result {idx} (score={doc.metadata.get('_score'):.4f}) ---")
    print(doc.page_content[:400], "...")
    print({k: v for k, v in doc.metadata.items() if not k.startswith('_')})
    print()

### Generate Answers with Retrieval-augmented Context

Once the decrypted documents are retrieved, we can use an LLM (e.g. OpenAI's GPT) to generate answers based on the retrieved documents.

In this example, we use the gemma3:270m model running locally with ollama.

In [ ]:
import requests

def generate_answer(docs, query, model="gemma3:270m"):
    instruction = "You are an assistant that answers questions based on the provided documents."
    prompt = f"""{instruction}:\n\n[Documents]\n"""
    for doc in docs:
        prompt += f"- {doc}\n"
    prompt += f"\n[Question]\n{query}\n[Answer]\n"

    response = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": model,
            "messages": [
                {"role": "system", "content": instruction},
                {"role": "user", "content": prompt}
            ],
            "stream": False
        }
    )
    response.raise_for_status()
    return response.json()["message"]["content"].strip()

In [ ]:
# Example usage
answer = generate_answer(results[0].metadata, query)
print(f"Generated Answer: \n{answer}")

### Clean Up

We can delete the created index and the registered key when they are no longer needed.

In [ ]:
store.client.ev.drop_index(index_name)

In [ ]:
store.client.ev.delete_key(key_id)